In [3]:
# %pip install --upgrade langchain-together

In [4]:
import os
import textwrap
import math
import pandas as pd
import numpy as np

from datetime import datetime
from dateutil.relativedelta import relativedelta

from langchain.schema.messages import AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_together import ChatTogether

import google.generativeai as genai
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory

from IPython.display import display
from IPython.display import Markdown

model = "meta-llama/Meta-Llama-3.1-405B-Instruct-Turbo"
temperature = 1
top_p = 0.9
seed = 0

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

llm = ChatTogether(
                together_api_key = os.getenv('TOGETHER_API_KEY'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
            )

# 定義策略元素字典，每個元素對應一個代號
elements = {
    "E1": "企業創新策略：企業創新的核心主題，涵蓋所有驅動企業增長和競爭力提升的策略。",
    "E2": "滿足客戶需求：企業創新的首要驅動力，能否滿足客戶需求直接影響創新成敗。",
    "E3": "吸引投資者關注：投資者的關注和信心能為企業提供必要的資金和支持，是創新成功的重要標誌。",
    "E4": "市場需求與趨勢：市場需求的變化和行業趨勢是影響企業股票表現的關鍵因素，企業需適應並預測市場變化。",
    "E5": "財務表現：穩定且增長的收入、利潤和現金流是吸引投資者的主要因素，能增加投資者信心，推動股價上漲。",
    "E6": "技術進步與研發投入：持續的技術創新和研發投入能提高企業競爭力和市場地位，促進企業長期增長。",
    "E7": "經營效率：有效的經營管理能提高企業盈利能力，進而提升股票價值。",
    "E8": "品牌聲譽與市場認可度：良好的品牌聲譽和市場認可度能增加企業產品的市場需求，促進銷售增長。",
    "E9": "政策與法規支持：政府政策和法規對行業的支持或限制會直接影響企業的發展前景和市場表現。",
    "E10": "競爭優勢與市場份額：企業在市場中的競爭優勢和市場份額的提升能增強其市場地位，吸引更多投資者。",
    "E11": "全球化與國際市場拓展：進入國際市場並擴展全球業務能增加企業的市場規模和收入來源。",
    "E12": "風險管理與應對能力：有效的風險管理策略和應對能力能降低企業運營風險，提高投資者信心。",
    "E13": "人才管理與員工激勵：擁有高素質的人才和有效的激勵機制能提升企業創新能力和生產效率。",
    "E14": "合作夥伴關係：與其他企業建立戰略合作夥伴關係能共享資源和技術，提升市場競爭力。",
    "E15": "市場營銷與品牌推廣：有效的市場營銷和品牌推廣策略能提高企業知名度和市場需求。",
    "E16": "客戶服務與滿意度：高質量的客戶服務和較高的客戶滿意度能提高客戶忠誠度和口碑。",
    "E17": "供應鏈管理：高效的供應鏈管理能確保產品質量和交付效率，降低成本。",
    "E18": "產品多樣化與創新：不斷推出創新產品和服務能滿足不同市場需求，提升企業競爭力。",
    "E19": "環保與可持續發展：採用環保和可持續發展策略能提高企業形象，吸引關注可持續投資的投資者。",
    "E20": "數位轉型與科技應用：積極進行數位轉型和應用新技術能提高運營效率和市場響應速度。",
    "E21": "競爭對手分析與市場擴展：深入分析競爭對手的策略和行動，制定有效的市場擴展和新市場進入策略。",
    "E22": "資本結構、財務穩定性與資金籌措：健康的資本結構和財務穩定性以及有效的資金籌措能增強企業的抗風險能力和持續發展能力。",
    "E23": "客戶數據分析：利用大數據和數據分析技術深入了解客戶需求，提升產品和服務的針對性。",
    "E24": "知識產權保護：有效保護企業的知識產權能防止競爭對手抄襲，提高創新成果的市場價值。",
    "E25": "企業文化與價值觀：建立積極向上的企業文化和價值觀能吸引優秀人才，提升企業凝聚力和創新力。",
    "E26": "股東回報與分紅政策：穩定的股東回報和合理的分紅政策能吸引長期投資者。",
    "E27": "社會責任與公益活動：積極參與社會責任和公益活動能提升企業形象，獲得更多社會支持。",
    "E28": "成本控制: 有效的成本控制能提高企業盈利能力。",
}

/Users/yanyifu/Documents/_Coding/LLM Predict Stock/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/yanyifu/Documents/_Coding/LLM Predict Stock/.venv/lib/python3.11/site-packages/langchain_core/utils/utils.py:225: UserWarning: WARNING! top_p is not default parameter.
                top_p was transferred to model_kwargs.
                Please confirm that top_p is what you intended.
  warnings.warn(
/Users/yanyifu/Documents/_Coding/LLM Predict Stock/.venv/lib/python3.11/site-packages/langchain_core/utils/utils.py:225: UserWarning: WARNING! seed is not default parameter.
                seed was transferred to model_kwargs.
                Please confirm that seed is what you intended.
  warnings.warn(


In [14]:
# 適應度函數

result = llm.invoke("\n權值股的表現往往會影響大盤的走勢，以下是台灣股市權值股的新聞標題：\n『權值股的熱門新聞標題['蘇姿丰開講 台塑王文淵到場聆聽：很期待AI時代到來', '台塑王文淵帶3董座聽蘇媽開講 直言：期待AI時代到來', 'AMD執行長蘇姿丰開講 王文淵率台塑三寶力挺、黃崇仁也現身', '蘇姿丰開講 台塑大家長王文淵率台塑三寶力挺、黃崇仁連兩日現身', '蘇姿丰COMPUTEX談AI時代 台塑王文淵親率集團高層聆聽', '蘇姿丰演講台廠力挺 台塑三寶、力積電期待AI時代到來', '超微大秀AI肌肉 董座蘇姿丰登台開講', '《國際金融》莫迪勝券在望 印度股市創新高', '外海疑有140億桶「巨大油田」南韓宣佈啟動探鑽', '超微變心轉單三星？ 蘇姿丰：與台積電有3奈米產品合作'] \n』\n\n如果有滿足以下的條件，則請只回答#yes 否則只回答#no\n\n\n用以下格式回答，你只需要綜合所有的新聞給出一次的回答就好：\n判斷結果: #yes/#no\n你的理由: 說明你的判斷理由\n")
print(result)

# # 評分方法
# start_day = datetime(2024, 6, 1)
# end_day = datetime(2024, 7, 15)
# stock_ids = ['2308', '2317', '2330', '2382', '2412', '2454', '2881', '2882', '2891', '6505']

# outpath = step1(stock_ids, start_day, end_day, prompt, "")



content='判斷結果: #yes\n理由：新聞標題中提到台塑（台塑王文淵）、AMD執行長蘇姿丰、台積電等權值股的新聞，且這些新聞內容與公司高層的表態、戰略合作和產品合作等相關消息都可能對股市產生影響，從而左右大盤的走勢。' response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 368, 'total_tokens': 453}, 'model_name': 'meta-llama/Meta-Llama-3.1-405B-Instruct-Turbo', 'system_fingerprint': None, 'finish_reason': 'eos', 'logprobs': None} id='run-184900ad-c079-4ccb-b405-15c0eeaed284-0' usage_metadata={'input_tokens': 368, 'output_tokens': 85, 'total_tokens': 453}
